# Đồ án KHDL cuối kỳ — Chuyên đề 3: Phân tích và gợi ý bất động sản TP.HCM

## Notebook 01: Mô tả bài toán và dữ liệu

- **Học viên**: [Điền tên]
- **Môn**: Lập trình cho Khoa học Dữ liệu
- **Giảng viên hướng dẫn**: [Điền tên giảng viên]
- **Ngày**: 23/07/2026

## 1. Bối cảnh và mục tiêu

Tin đăng bất động sản nhà phố tại TP.HCM thường có:
- **Giá viết không thống nhất** ("3,5 tỷ", "3500 triệu", "Giá thỏa thuận").
- **Diện tích dạng số thập phân với dấu phẩy** ("75,5 m²").
- **Địa danh viết tắt / lẫn lộn** ("Q.1", "Quận 1", "1" — đều là cùng một quận).
- **Nhiều tin đăng trùng** và **ngoại lệ rõ ràng** (giá vài chục triệu, diện tích 7000 m²).

Mục tiêu đồ án:
1. Xây dựng **pipeline chuẩn hóa** giá, diện tích, giá/m², địa danh và thuộc tính mô tả.
2. Phát hiện **tin trùng** và **ngoại lệ**.
3. **Dự đoán** `price_per_m2` từ các đặc trưng (diện tích, phòng ngủ, hướng, quận, tiện ích).
4. **Phân khúc** BĐS bằng K-Means và **gợi ý top 5** theo hồ sơ nhu cầu (ngân sách, số phòng, diện tích, khu vực).

## 2. Sáu câu hỏi nghiên cứu

1. Khu vực nào có giá trên mét vuông (`price_per_m2`) cao nhất?
2. Diện tích và số phòng ảnh hưởng như thế nào đến tổng giá và giá/m²?
3. Hướng nhà, vị trí quận có liên hệ gì với giá không?
4. Tin nào có giá/diện tích bất thường hoặc có khả năng trùng?
5. Mô hình dự đoán giá/m² đạt sai số bao nhiêu (MAE / RMSE / R²)?
6. Top 5 tin nào phù hợp với từng hồ sơ nhu cầu (gia đình trẻ / nhà đầu tư / mua đầu tiên)?

## 3. Hai nguồn dữ liệu

| Nguồn | File | Số dòng | Mô tả |
|---|---|---|---|
| 1 (chính) | `data/raw/real_estate_with_price_per_m2.csv` | 3000 | Tin đăng BĐS nhà phố tại TP.HCM, đã crawl từ nguồn alonhadat-style, có schema 20 cột. |
| 2 (phụ) | `data/raw/neighborhood_amenities.csv` | 100 | Thông tin tiện ích (trường học, bệnh viện, siêu thị, công viên, bus) theo (quận, phường) — tự tổng hợp deterministic. |

## 4. Từ điển dữ liệu (rút gọn)

| Cột | Kiểu | Mô tả |
|---|---|---|
| `listing_id` | int | Mã tin đăng. |
| `title`, `description` | str | Tiêu đề + mô tả. |
| `property_type` | str | Loại hình (toàn bộ là "Nhà" trong dataset này). |
| `province` | str | Tỉnh/TP (toàn bộ là "Hồ Chí Minh"). |
| `district` | str | Quận/huyện (thô — có dạng số "1" và tên "Bình Thạnh"). |
| `ward` | str | Phường/xã (có missing ~6.7%). |
| `street_name`, `project_name` | str | Tên đường, tên dự án (missing ~26.5% / 98.6%). |
| `total_price` | float | Tổng giá (VND). |
| `area_m2` | float | Diện tích (m²). |
| `price_per_m2` | float | Giá/m² tính sẵn (sẽ tính lại để đảm bảo nhất quán). |
| `floor_count`, `frontage_width`, `house_depth`, `road_width` | float | Đặc trưng nhà (nhiều missing). |
| `bedrooms`, `bathrooms` | float | Số phòng. |
| `house_direction` | str | Hướng nhà (8 hướng chính + dạng ghép). |
| `posted_at` | str | Ngày đăng (chuỗi ISO). |

Chi tiết từng cột xem `reports/data_dictionary.md`.

## 5. Dữ liệu mẫu — 5 dòng đầu

In [1]:
import pandas as pd

raw = pd.read_csv('data/raw/real_estate_with_price_per_m2.csv')
print('Shape:', raw.shape)
raw.head()

Shape: (3000, 20)


,listing_id,title,description,property_type,province,district,ward,street_name,project_name,total_price,area_m2,price_per_m2,floor_count,frontage_width,house_depth,road_width,bedrooms,bathrooms,house_direction,posted_at
0,1,Nhà hai mặt tiền ngay chợ Hoà Hưng,"Bán nhà 2 Mặt tiền 557 Cách Mạng Tháng Tám, DT...",Nhà,Hồ Chí Minh,10,15,Cách Mạng Tháng Tám,NaN,2.600000e+10,88.0,2.954545e+08,5.0,4.0,NaN,NaN,NaN,NaN,Tây Nam,2025-06-18 09:45:44.494
1,2,"Nhà hẻm kinh doanh 10m thông, 3 lầu 80m2 5x15,...","HẺM KINH DOANH XE TẢI THÔNG 10M- NHÀ 2 LẦU, PH...",Nhà,Hồ Chí Minh,Tân Bình,15,Phan Huy Ích,NaN,8.000000e+09,80.0,1.000000e+08,NaN,5.0,NaN,NaN,4.0,3.0,Đông,2025-06-11 13:24:44.488
2,3,"Bán NR 4PN, 5WC, 80m2 tại Lê Văn Thọ, Gò Vấp, ...","Nhà riêng 4 phòng ngủ, 5 toilet, diện tích 80m...",Nhà,Hồ Chí Minh,Gò Vấp,11,Lê Văn Thọ,NaN,7.200000e+09,80.0,9.000000e+07,4.0,4.0,NaN,NaN,4.0,5.0,Tây,2025-06-18 10:52:44.369
3,4,"Bán nhà 4 tầng Phan Huy Ích, P. 12, Gò Vấp đườ...",Bán nhà 4 tầng Phan Huy Ích P. 12 Gò Vấp đường...,Nhà,Hồ Chí Minh,Gò Vấp,12,Phan Huy Ích,NaN,6.500000e+09,48.0,1.354167e+08,NaN,4.2,NaN,NaN,5.0,5.0,Tây Nam,2025-06-18 17:32:59.686
4,5,Căn Duy Nhất trên tuyến đường Trần Hưng Đao P....,Hàng hiếm trên thị trường - Mặt tiền Trần Hưng...,Nhà,Hồ Chí Minh,1,Cầu Kho,NaN,NaN,4.190000e+10,80.0,5.237500e+08,NaN,4.6,NaN,NaN,NaN,NaN,Tây Bắc,2025-06-21 09:24:37.408


## 6. Dữ liệu mẫu — 5 dòng cuối

In [2]:
raw.tail()

,listing_id,title,description,property_type,province,district,ward,street_name,project_name,total_price,area_m2,price_per_m2,floor_count,frontage_width,house_depth,road_width,bedrooms,bathrooms,house_direction,posted_at
2995,2996,"Nhà đẹp Phạm Thế Hiển. Phường 4. Quận 8. DT 3,...",#PTH.B?s9.5 Mô Tả: Nhà đẹp mới tinh vào ờ ngay...,Nhà,Hồ Chí Minh,8,4,Phạm Thế Hiển,NaN,7.600000e+09,51.8,1.467181e+08,NaN,4.0,NaN,NaN,4.0,3.0,Bắc,2025-06-07 04:15:35.580
2996,2997,15ty2 thương lượng bán nhà mặt tiền đường nhựa...,15ty thương lượng chính chủ định cư cần bán gấ...,Nhà,Hồ Chí Minh,8,4,NaN,NaN,1.520000e+10,110.0,1.381818e+08,NaN,4.0,NaN,NaN,5.0,NaN,Tây Bắc,2025-06-09 07:22:55.952
2997,2998,"Giá tốt, 1 trệt 1 lầu 3.6 tỷ 80m2, 2 phòng ngủ...",80m2 full thổ cư ko quy hoạch lộ giới Gần chợ ...,Nhà,Hồ Chí Minh,Thủ Đức,Long Trường,NaN,NaN,3.600000e+09,80.0,4.500000e+07,NaN,6.2,NaN,NaN,2.0,3.0,Đông Nam,2025-06-20 03:55:08.826
2998,2999,"Bán nhà mới đẹp 70m2 - chỉ: 3,95 tỷ - 4x18m - ...",Mặt tiền đường lớn - nhà mới - hàng xóm khu đô...,Nhà,Hồ Chí Minh,Thủ Đức,Hiệp Bình Phước,NaN,NaN,3.950000e+09,70.0,5.642857e+07,NaN,4.0,NaN,NaN,2.0,2.0,Tây,2025-06-11 05:41:30.848
2999,3000,Nhà cấp 4 hẻm xe hơi 5m đường số 79 Đỗ Xuân Hợ...,"Bán nhà hẻm xe hơi đường 79, phường Phước Long...",Nhà,Hồ Chí Minh,Thủ Đức,Phước Long B,NaN,NaN,4.500000e+09,55.7,8.078995e+07,NaN,4.0,NaN,NaN,1.0,1.0,Tây Nam,2025-06-02 04:48:18.039


## 7. Schema và dtype

In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   listing_id       3000 non-null   int64  
 1   title            3000 non-null   str    
 2   description      3000 non-null   str    
 3   property_type    3000 non-null   str    
 4   province         3000 non-null   str    
 5   district         2984 non-null   str    
 6   ward             2800 non-null   str    
 7   street_name      2205 non-null   str    
 8   project_name     42 non-null     str    
 9   total_price      2974 non-null   float64
 10  area_m2          3000 non-null   float64
 11  price_per_m2     2974 non-null   float64
 12  floor_count      764 non-null    float64
 13  frontage_width   2756 non-null   float64
 14  house_depth      13 non-null     float64
 15  road_width       460 non-null    float64
 16  bedrooms         2491 non-null   float64
 17  bathrooms        2428 non

## 8. Tỉ lệ missing theo cột

In [4]:
(raw.isna().mean() * 100).round(2).sort_values(ascending=False)

house_depth        99.57
project_name       98.60
road_width         84.67
floor_count        74.53
street_name        26.50
bathrooms          19.07
bedrooms           16.97
frontage_width      8.13
ward                6.67
total_price         0.87
price_per_m2        0.87
district            0.53
listing_id          0.00
house_direction     0.00
area_m2             0.00
title               0.00
province            0.00
property_type       0.00
description         0.00
posted_at           0.00
dtype: float64

## 9. Kết luận

Dữ liệu đủ lớn (3000 dòng) và đa dạng (22 quận/huyện, 144 phường/xã, nhiều mức giá).
Các cột có missing > 80% (`house_depth`, `project_name`, `road_width`) sẽ **không** dùng làm feature cho mô hình, chỉ giữ để tham khảo khi cần.

Tiếp theo: mở `02_collection_and_cleaning.ipynb` để xem pipeline làm sạch chi tiết.